# Problem 2: Comparison of Perceptual Quality Measures

We compare PSNR and SSIM (luminance-only, contrast-only, structure-only, overall) against human perception scores (DMOS) using the **Spearman Rank Order Correlation Coefficient (SROCC)**.

In [ ]:
import os
import numpy as np
from PIL import Image
import scipy.io
from scipy.stats import spearmanr
from scipy.ndimage import gaussian_filter
# import warnings
# warnings.filterwarnings('ignore')

## Load Dataset Metadata

In [4]:
# Paths
DATASET_DIR = os.path.join('..', 'dataset', 'hw5')
GBLUR_DIR   = os.path.join(DATASET_DIR, 'gblur')
REFIMGS_DIR = os.path.join(DATASET_DIR, 'refimgs')
MAT_PATH    = os.path.join(DATASET_DIR, 'hw5.mat')

# Load .mat file
mat = scipy.io.loadmat(MAT_PATH)

blur_dmos   = mat['blur_dmos'].flatten()           # shape (174,)
blur_orgs   = mat['blur_orgs'].flatten().astype(bool)
refnames    = [mat['refnames_blur'][0, i].item() for i in range(174)]

print(f'Total entries    : {len(blur_dmos)}')
print(f'Original images  : {blur_orgs.sum()}')
print(f'Distorted images : {(~blur_orgs).sum()}')
# print(f'Sample refnames  : {refnames[:5]}')

Total entries    : 174
Original images  : 29
Distorted images : 145


## SSIM Helper Functions

SSIM decomposes into three components:
$$
l(x,y) = \frac{2\mu_x\mu_y + C_1}{\mu_x^2 + \mu_y^2 + C_1},\quad
c(x,y) = \frac{2\sigma_x\sigma_y + C_2}{\sigma_x^2 + \sigma_y^2 + C_2},\quad
s(x,y) = \frac{\sigma_{xy} + C_3}{\sigma_x\sigma_y + C_3}
$$
with $C_1=(K_1 L)^2$, $C_2=(K_2 L)^2$, $C_3 = C_2/2$ and overall $\text{SSIM} = l \cdot c \cdot s$.


In [ ]:
# https://arxiv.org/pdf/2006.13846
def compute_ssim_components(img_ref, img_dist, radius = 5, sigma = 1.5, K1=0.01, K2=0.03, L=255):
    """
    Compute per-pixel SSIM and its three components using a sliding window.
    Returns (luminance_map, contrast_map, structure_map, ssim_map).
    """
    C1 = (K1 * L) ** 2
    C2 = (K2 * L) ** 2
    C3 = C2 / 2.0

    x = img_ref.astype(np.float64)
    y = img_dist.astype(np.float64)

    mu_x = gaussian_filter(x, sigma, radius = radius)
    mu_y = gaussian_filter(y, sigma, radius = radius)

    mu_x_squared = mu_x ** 2
    mu_y_squared = mu_y ** 2

    sigma_x_squared = gaussian_filter(x * x, sigma, radius = radius) - mu_x_squared
    sigma_y_squared = gaussian_filter(y * y, sigma, radius = radius) - mu_y_squared
    sigma_xy = gaussian_filter(x * y, sigma, radius = radius) - mu_x * mu_y

    sigma_x  = np.sqrt(np.maximum(sigma_x_squared, 0))
    sigma_y  = np.sqrt(np.maximum(sigma_y_squared, 0))

    l_map = (2 * mu_x * mu_y + C1) / (mu_x_squared + mu_y_squared + C1)
    c_map = (2 * sigma_x * sigma_y + C2) / (sigma_x_squared + sigma_y_squared + C2)
    s_map = (sigma_xy + C3) / (sigma_x * sigma_y + C3)

    ssim_map = l_map * c_map * s_map

    return l_map.mean(), c_map.mean(), s_map.mean(), ssim_map.mean()


def compute_psnr(img_ref, img_dist, L=255):
    """
    Peak Signal-to-Noise Ratio in the pixel domain.
    """
    x = img_ref.astype(np.float64)
    y = img_dist.astype(np.float64)
    mse = np.mean((x - y) ** 2)
    if mse == 0:
        return np.inf
    return 10 * np.log10(L ** 2 / mse)


def load_gray(path):
    """Load image and convert to grayscale luminance channel (Y in YCbCr)."""
    img = Image.open(path).convert('YCbCr')
    y_channel = np.array(img)[:, :, 0]
    return y_channel

## Compute Metrics for All Images

In [10]:
psnr_scores  = []   
luminance_scores   = []  # SSIM luminance component
contrast_scores   = []  # SSIM contrast component
structure_scores   = []  # SSIM structure component
ssim_scores  = []  # Overall SSIM

for i in range(len(blur_dmos)):
    dist_path = os.path.join(GBLUR_DIR, f'img{i+1}.bmp')
    ref_path  = os.path.join(REFIMGS_DIR, refnames[i])

    dist_img = load_gray(dist_path)
    ref_img  = load_gray(ref_path)

    # Match sizes (crop ref to dist if needed)
    h = min(dist_img.shape[0], ref_img.shape[0])
    w = min(dist_img.shape[1], ref_img.shape[1])
    dist_img = dist_img[:h, :w]
    ref_img  = ref_img[:h, :w]

    p = compute_psnr(ref_img, dist_img)
    l, c, s, ssim = compute_ssim_components(ref_img, dist_img)

    psnr_scores.append(p)
    luminance_scores.append(l)
    contrast_scores.append(c)
    structure_scores.append(s)
    ssim_scores.append(ssim)

psnr_scores = np.array(psnr_scores)
luminance_scores  = np.array(luminance_scores)
contrast_scores  = np.array(contrast_scores)
structure_scores  = np.array(structure_scores)
ssim_scores = np.array(ssim_scores)

print(f'Completed computing metrics for {len(blur_dmos)} images.')

Completed computing metrics for 174 images.


In [8]:
psnr_scores

array([31.70328708, 26.29263967, 25.72850478, 39.15839181, 22.08117259,
       23.79542691, 24.29276295, 25.15346024, 34.3116253 , 20.79576709,
       18.02045634, 35.17595754, 30.02504965, 28.04078687, 25.94103926,
       22.8417074 , 20.5459908 , 25.29527415, 25.10893786, 24.07545978,
       24.29217638, 25.99232443, 31.67795677, 18.51346994, 18.60704112,
       23.56225443, 28.33976453, 36.19430085, 29.96051724, 26.43308044,
       28.49463988, 34.87729635, 22.78042602, 27.90361105, 29.41211834,
       28.30573912, 23.15541348, 22.23026016, 26.0923572 , 20.50464125,
       19.8134465 , 29.53440219, 30.05885757, 27.90845599, 29.40702065,
       32.21113322, 25.45229542, 26.12252606, 29.50411414, 23.23244938,
       29.76765119, 21.6976271 , 25.01635392, 26.25300466, 26.06434049,
       31.47214968, 28.38357872, 19.5711175 , 25.57253727, 30.10237607,
       31.66401651, 23.52388923, 35.79075745, 27.20516286, 23.11040477,
       22.81814419, 35.0293959 , 23.51691268, 24.1046911 , 25.23

In [9]:
ssim_scores

array([0.88357849, 0.87563934, 0.86487654, 0.98708958, 0.71201445,
       0.75895798, 0.82522375, 0.88409803, 0.95828674, 0.64767995,
       0.62924447, 0.9527128 , 0.94182916, 0.88632617, 0.78470542,
       0.64477027, 0.40854807, 0.81398472, 0.72192659, 0.77391451,
       0.84228757, 0.76620209, 0.95661881, 0.50260004, 0.45266699,
       0.71990811, 0.91135027, 0.95424518, 0.88374167, 0.81910819,
       0.85934278, 0.96955656, 0.79122848, 0.8138207 , 0.88441955,
       0.75018518, 0.79578576, 0.6478647 , 0.83476206, 0.47794231,
       0.56425468, 0.79697486, 0.88217022, 0.91246199, 0.94792843,
       0.94146129, 0.79231776, 0.77290446, 0.83575765, 0.68983596,
       0.95917681, 0.65550743, 0.74154514, 0.75273608, 0.79055114,
       0.91216171, 0.74035232, 0.32542087, 0.83381161, 0.941012  ,
       0.86615653, 0.79144089, 0.96976831, 0.83131967, 0.8293073 ,
       0.73866103, 0.97914781, 0.59937141, 0.78166028, 0.73314415,
       0.99065297, 0.83860299, 0.44525232, 0.76547556, 0.82947

## Part C.1: Spearman Rank Order Correlation Coefficient

Remove entries where `blur_orgs == 1` (original images), leaving 145 distorted images, then compute SROCC with DMOS scores.

In [13]:
# Keep only non-original (distorted) images
mask = ~blur_orgs  # True where image is distorted

dmos_filtered = blur_dmos[mask]
psnr_filtered = psnr_scores[mask]
lum_filtered = lum_scores[mask]
con_filtered = con_scores[mask]
str_filtered = str_scores[mask]
ssim_filtered = ssim_scores[mask]

print(f'Number of distorted images (after removing originals): {mask.sum()}')

# Compute Spearman correlations
srocc_psnr,  _ = spearmanr(dmos_filtered, psnr_filtered)
srocc_lum,   _ = spearmanr(dmos_filtered, lum_filtered)
srocc_con,   _ = spearmanr(dmos_filtered, con_filtered)
srocc_str,   _ = spearmanr(dmos_filtered, str_filtered)
srocc_ssim,  _ = spearmanr(dmos_filtered, ssim_filtered)

print('\n=== Spearman Rank Order Correlation Coefficient ===')
print(f'  PSNR                   : {srocc_psnr:.4f}')
print(f'  SSIM (luminance only)  : {srocc_lum:.4f}')
print(f'  SSIM (contrast only)   : {srocc_con:.4f}')
print(f'  SSIM (structure only)  : {srocc_str:.4f}')
print(f'  SSIM (overall)         : {srocc_ssim:.4f}')

Number of distorted images (after removing originals): 145

=== Spearman Rank Order Correlation Coefficient ===
  PSNR                   : -0.7821
  SSIM (luminance only)  : -0.9329
  SSIM (contrast only)   : -0.9063
  SSIM (structure only)  : -0.9053
  SSIM (overall)         : -0.9028


## Part C.2: Comment on Relative Performances

Higher |SROCC| indicates stronger monotonic correlation with human perception (DMOS).

**Key observations:**

- **PSNR** measures distortion in pixel space without accounting for perceptual sensitivity — it typically shows moderate correlation with human perception.
- **SSIM (luminance)** captures mean brightness differences, which humans are fairly sensitive to but which blur doesn't affect strongly (mean brightness is preserved after Gaussian blur).
- **SSIM (contrast)** captures variance differences between reference and distorted patches — blurring significantly reduces local variance, so this term is expected to be more sensitive.
- **SSIM (structure)** captures correlation of local patterns. Gaussian blur destroys fine structure, so this term is expected to have the highest sensitivity.
- **Overall SSIM** is the product of all three terms and generally provides the best correlation with human opinion scores because it jointly models luminance, contrast, and structure distortion.

In summary, SSIM (overall) is expected to outperform PSNR, while among the SSIM components, the **structure** and **contrast** terms contribute most to perceptual correlation for Gaussian-blurred images.